# Alineando LLMs con actividad neural

¿Qué tan parecidas son las representaciones internas de un modelo de lenguaje a las que el cerebro construye al escuchar una historia? En este cuaderno exploramos esa pregunta con un **modelo de codificación** (*encoding model*): usamos las activaciones de modelos de lenguaje como predictores de la actividad cerebral medida con fMRI, y evaluamos qué tan bien la predicen en distintas regiones.

El trabajo se basa en Bonnasse-Gahot y Pallier (2024), quienes muestran que, a medida que aumenta la complejidad de los modelos de lenguaje, sus predicciones recuperan la lateralización izquierda característica del procesamiento del lenguaje en el cerebro.

In [1]:
# Descomentá y corré esta línea si estas corriendo en Google Colab
# !pip install nilearn remotezip

Como es usual, comenzamos importando las librerías que vamos a utilizar:

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy as sp
import pathlib
import sklearn
import nilearn
import remotezip

A lo largo de este practico vamos a usar varios archivos, asegúrese de subirlos a todos a su entorno de ejecución. El siguiente archivo verifica que puede encontrarlos a todos.

In [3]:
import pathlib
import urllib

base_url = "https://raw.githubusercontent.com/MaestriaCienciasCognitivas/ncc/main/book/static/"
archivos = ['Practico12_mask_lpp_en.nii', 'Practico12_word_information.csv', 'Practico12_reliable_voxels.npy']

CARPETA = Path("data")
CARPETA.mkdir(exist_ok=True)

for nombre in archivos:
    ruta = CARPETA / nombre
    if ruta.exists():
        print(f"{nombre}: ya descargado, se omite")
        continue
    urllib.request.urlretrieve(base_url + nombre, ruta)
    print(f"{nombre}: descargado")

NameError: name 'Path' is not defined

## *GloVe*

Ahora bien, un **word embedding** es una representación numérica de palabras en un espacio vectorial de baja dimensión, usada en procesamiento de lenguaje natural. Captura relaciones semánticas y sintácticas, donde palabras similares tienen vectores cercanos. Se generan mediante modelos como Word2Vec, GloVe o FastText, y son útiles para tareas como clasificación de texto, búsqueda semántica y análisis de similitud. Ejemplo: king - man + woman ≈ queen.

A continuación, cargamos los datos experimentales y trataremos de entender cómo están codificados.

In [ ]:
import pandas as pd
word_information = pd.read_csv(CARPETA / 'Practico12_word_information.csv')
display(word_information.head())

**GloVe** (*Global Vectors for Word Representation*, Pennington, Socher y Manning, 2014) es uno de los métodos clásicos para construir estos vectores. Su idea central es que las proporciones de co-ocurrencia entre palabras contienen información semántica: si dos palabras aparecen juntas con frecuencia, o si se relacionan de forma parecida con otras palabras, comparten algún aspecto de significado.

El proceso comienza con un corpus grande de texto. Se recorre ese corpus y se construye una matriz $X$, donde $X_{ij}$ cuenta cuántas veces la palabra $j$ aparece en la vecindad de la palabra $i$, dentro de una ventana de contexto de unas 10 a 15 palabras. Las palabras más cercanas suelen pesar más que las lejanas.

GloVe asigna a cada palabra un vector $w_i$ y un vector de contexto $\tilde{w}_j$, junto con dos sesgos escalares $b_i$ y $\tilde{b}_j$. El objetivo es que el producto escalar más los sesgos aproxime el logaritmo del conteo de co-ocurrencia:

$$
w_i^\top \tilde{w}_j + b_i + \tilde{b}_j \approx \log X_{ij}
$$

Este ajuste se logra minimizando una suma de errores cuadráticos ponderados:

$$
J = \sum_{i,j} f(X_{ij})\left(w_i^\top \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2
$$

donde la función de ponderación es

$$
f(x) =
\begin{cases}
\left(\dfrac{x}{x_{\max}}\right)^{\alpha} & \text{si } x < x_{\max} \\
1 & \text{en otro caso}
\end{cases}
$$

En el paper original se usan $x_{\max} = 100$ y $\alpha = 0.75$. Esta ponderación evita que los pares muy frecuentes, como "the" con cualquier otra palabra, dominen el entrenamiento, y también evita que los pares que nunca co-ocurren aporten ruido. La optimización se hace con un método de gradiente descendente, típicamente AdaGrad, durante decenas de iteraciones.

Al terminar, cada palabra tiene dos vectores. En la práctica se suele usar la suma $w_i + \tilde{w}_i$ como representación final. La dimensión del vector, por ejemplo 300, es un hiperparámetro que se elige según el compromiso entre calidad y costo.

Los vectores que se distribuyen como `glove.6B` se entrenaron con unos 6 mil millones de tokens de Wikipedia 2014 y del corpus Gigaword 5 de noticias, con un vocabulario de alrededor de 400 mil palabras en minúscula. Por eso reflejan el uso del inglés en enciclopedias y noticias.

En el código a continuación descargamos los *embeddings* de GloVe y, para cada palabra en nuestro conjunto de datos, nos quedamos con su *embedding* correspondiente en GloVe.

In [ ]:
import os
import zipfile
import huggingface_hub

txt_path = CARPETA / "glove.6B.300d.txt"

if not txt_path.exists():
    zip_path = huggingface_hub.hf_hub_download(
        repo_id="stanfordnlp/glove", filename="glove.6B.zip", repo_type="model"
    )
    with zipfile.ZipFile(zip_path) as z:
        z.extract("glove.6B.300d.txt", path=CARPETA)

palabras = word_information['word'].astype(str).str.lower()
necesarias = set(palabras)

glove_vectores = {}
with open(txt_path, encoding="utf8") as f:
    for line in f:
        partes = line.rstrip().split(" ")
        if partes[0] in necesarias:
            glove_vectores[partes[0]] = np.asarray(partes[1:], dtype=np.float32)

dim = 300
emb = np.zeros((len(palabras), dim), dtype=np.float32)
en_vocab = palabras.isin(glove_vectores.keys()).to_numpy()
for i, w in enumerate(palabras):
    if w in glove_vectores:
        emb[i] = glove_vectores[w]

secciones = sorted(word_information['section'].unique())
glove = np.empty(len(secciones), dtype=object)
for i, s in enumerate(secciones):
    mask = (word_information['section'] == s).to_numpy()
    glove[i] = emb[mask]

print("Variable: glove")
print(f"N={glove.shape[0]}")
for run in range(glove.shape[0]):
    print(f"Tamaño de la matriz n={run} de activaciones: {glove[run].shape}")

La primera dimensión se corresponde con el número de embeddings recogidos en cada sesión, mientras que la segunda dimensión es la cantidad de dimensiones de ese embedding.

1. ¿Por qué el número de embeddings cambia entre cada sesión?
2. ¿Qué es una dimensión?

Veamos como luce un embedding en una determinada sesión:

In [ ]:
run = 0
word = 1500
embedding = glove[run][word]

fig, axs = plt.subplot_mosaic([['activations', 'histogram']], layout='constrained', figsize=(6, 3))
axs["activations"].matshow(embedding.reshape(20, 15), cmap="Spectral_r")
axs["activations"].set_title("Embedding")
axs["activations"].tick_params(labeltop=False, labelleft=False)
axs["histogram"].hist(embedding)
axs["histogram"].set_title("Histograma")
plt.show()

Antes de avanzar, necesitamos asignar un instante temporal a cada palabra. Por suerte, tenemos información temporal de los embeddings. Esto es, para cada palabra, sabemos en que tiempo comenzó y terminó. Cargemos estos datos y veamos como lucen:

In [ ]:
secciones = sorted(word_information['section'].unique())
timestamps = np.empty(len(secciones), dtype=object)
for i, s in enumerate(secciones):
    timestamps[i] = word_information.loc[word_information['section'] == s, ['onset', 'offset']].to_numpy()

print("Variable: timestamps")
n_runs = timestamps.shape[0]
print(f"N={n_runs}")
for run in range(n_runs):
    print(f"Tamaño de la matriz n={run} de marcas temporales: {timestamps[run].shape}")
print(f"Primeros 5 valores de n=0: {timestamps[run][:5]}")

La primera dimensión se corresponde con la cantidad de palabras en la sesión, mientras que la segunda dimensión contiene la marca temporal. Por ejemplo:

In [ ]:
run = 0
word = 0
timestamp = timestamps[run][word]

print(f"Inicio: {timestamp[0]}s\nFin: {timestamp[1]}s")

## BOLD

La **señal BOLD** (Blood Oxygen Level Dependent) es un tipo de señal medida en fMRI que refleja cambios en la oxigenación de la sangre asociados con la actividad cerebral. Cuando las neuronas se activan, aumenta el flujo sanguíneo local, disminuye la deoxihemoglobina y se genera un contraste detectable por la resonancia magnética. Esto permite mapear de forma indirecta qué áreas del cerebro están activas durante tareas o en reposo.

A continuación, cargamos los datos experimentales y trataremos de entender como están codificados:

In [ ]:
import gzip
import shutil
from pathlib import Path
from remotezip import RemoteZip

url = "https://zenodo.org/records/19097233/files/l-bg/llms_brain_lateralization-v1.0.0.zip?download=1"
destino = CARPETA / "lpp_en_average_subject"
destino.mkdir(parents=True, exist_ok=True)

nombres = [f"average_subject_run-{i}" for i in range(9)]

with RemoteZip(url) as z:
    infos = {Path(info.filename).name: info for info in z.infolist()
             if "lpp_en_average_subject/" in info.filename}

    for nombre in nombres:
        gz_path = destino / f"{nombre}.gz"
        salida = destino / f"{nombre}.joblib"
        
        info = infos[f"{nombre}.gz"]
        if gz_path.exists() and gz_path.stat().st_size == info.file_size:
            print(f"{gz_path.name}: ya descargado, se omite")
        else:
            with z.open(info) as src, open(gz_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"{gz_path.name}: descargado")
        
        if salida.exists():
            print(f"{salida.name}: ya descomprimido, se omite")
        else:
            with gzip.open(gz_path, "rb") as f_in, open(salida, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f"{salida.name}: descomprimido")

In [ ]:
import joblib

fmri = np.empty(9, dtype=object)
for i in range(9):
    fmri[i] = joblib.load(destino / f'average_subject_run-{i}.joblib')

n_runs = fmri.shape[0]
print(f"N={n_runs}")
for run in range(n_runs):
    print(f"Tamaño de la matriz n={run} de señales BOLD: {fmri[run].shape}")

La primera dimensión se corresponde con el número de escaneos recogidos en cada sesión, mientras que la segunda dimensión es la cantidad de vóxeles.

1. ¿Por qué el número de escaneos cambia entre cada sesión?
2. ¿Qué es un vóxel?

Veamos como luce una señal BOLD en una determinada sesión:

In [ ]:
run = 0
scan = 0
bold = fmri[run][scan]

fig, axs = plt.subplot_mosaic([['bold', 'histogram']], layout='constrained', figsize=(9, 3))
axs["bold"].matshow(bold.reshape(130, 199), cmap="Spectral_r")
axs["bold"].set_title("Señal BOLD")
axs["bold"].tick_params(labeltop=False, labelleft=False)
axs["histogram"].hist(bold)
axs["histogram"].set_title("Histograma")
plt.show()

Esta señal se encuentra muy desordenada debido a que aún no incorporamos información espacial de a que coordenada en el cerebro se corresponde cada vóxel.

Para ordenarla y visualizar mejor los vóxels podemos cargar una máscara que asocia cada vóxel a una coordenada 3D.

In [ ]:
from nilearn.maskers import NiftiMasker
from nilearn.plotting import plot_roi

masker = NiftiMasker(CARPETA / 'Practico12_mask_lpp_en.nii')
masker.fit()
n_voxels = masker.n_elements_
print(f"Número de vóxels: {n_voxels}")

plot_roi(masker.mask_img_, title="Máscara", cmap="Spectral_r", cut_coords=(0, 0, 0))
plt.show()

Esta máscara nos permite mejorar el código que visualizaba una señal BOLD determinada:

In [ ]:
from nilearn.plotting import plot_glass_brain

run = 0
scan = 2
bold = fmri[run][scan]

fig, axs = plt.subplot_mosaic([['bold', 'histogram'], ["nifti", "nifti"]], layout='constrained', figsize=(9, 6))
axs["bold"].matshow(bold.reshape(130, 199), cmap="Spectral_r")
axs["bold"].set_title("Señal BOLD")
axs["bold"].tick_params(labeltop=False, labelleft=False)
axs["histogram"].hist(bold)
axs["histogram"].set_title("Histograma")

plot_glass_brain(masker.inverse_transform(bold), threshold=0., vmin=0., cmap='Spectral_r', plot_abs=True, axes=axs["nifti"])
plt.show()

Reconstruir las marcas temporales de cada escaneo es fácil, ya que sabemos que la resolución temporal del resonador es de 2 segundos. Por lo tanto, construimos vectores para cada sesión con las marcas temporales de cada *scan*, siendo la marca temporal la del centro de la ventana de 2 segundos.

In [ ]:
t_r = 2.0
num_sessions = len(fmri)
fmri_timestamps = np.empty(num_sessions, dtype=object)
for run in range(num_sessions):
    fmri_timestamps[run] = np.arange(fmri[run].shape[0]) * t_r  + .5*t_r

n_runs = timestamps.shape[0]
print(f"N={n_runs}")
for run in range(n_runs):
    print(f"Tamaño de la matriz n={run} de marcas temporales: {fmri_timestamps[run].shape}")

print(f"Primeros 5 valores en n=0: {fmri_timestamps[run][:5]}")

Para una sesión, por ejemplo, la primera:

1. ¿Cuantos embeddings tenemos?
2. ¿Cuantas señales BOLD?
3. ¿Cómo podemos hacer para compararlas?

Con esta información, estamos en condiciones de transformar una serie de activaciones en señales BOLD esperadas. Para esto, usaremos un modelo computacional llamado [Glover](10.1006/nimg.1998.0419).

## Glover

**Glover** se refiere a un modelo de función de respuesta hemodinámica (HRF) ampliamente utilizado en análisis de datos de fMRI. Este modelo describe la relación temporal entre la actividad neuronal y la señal BOLD  registrada en imágenes funcionales del cerebro.

La **HRF** es una función matemática que modela cómo la señal BOLD cambia en respuesta a un evento neuronal. Cuando las neuronas se activan, el flujo sanguíneo local aumenta para satisfacer la demanda de oxígeno, lo que genera un cambio característico en la señal BOLD. Este cambio ocurre en tres fases principales:

1. Rápido incremento inicial (alrededor de 4-6 segundos después del evento).
2. Pico máximo.
3. Caída posterior (incluso una pequeña "infracción" o valor negativo por debajo de la línea base).

El modelo de Glover permite capturar las características principales del cambio BOLD observado en la práctica.

Cuando especificamos `hrf_model='glover'` en la función `compute_regressor`, se usa este modelo para convolucionar eventos y generar un regresor que represente la dinámica de la señal esperada en respuesta al diseño experimental.

A continuación, transformamos nuestra matriz de word embeddings en cada sesión, en otra matriz que captura la dinámica de la señal BOLD, de forma tal que podamos compararlas:

In [ ]:
from nilearn.glm.first_level import compute_regressor

def compute_regressor_from_activations(activations, timestamps, frame_times):
    """
    Convoluciona las activaciones de cada palabra con la HRF glover para armar los regresores de fMRI.

    Parameters
    ----------
    activations : ndarray de forma (n_eventos, n_neuronas)
        Amplitud de cada unidad para cada evento (palabra) de la corrida. Acá, cada
        "neurona" es una dimensión del embedding de la palabra (por ejemplo, 300 para GloVe).
    timestamps : ndarray de forma (n_eventos, 2)
        Onset (columna 0) y offset (columna 1) de cada evento, en segundos.
        La duración de cada evento se calcula como offset - onset.
    frame_times : ndarray de forma (n_escaneos,)
        Momento de cada volumen de fMRI, en segundos, ubicado en el centro de su
        ventana de adquisición .

    Returns
    -------
    regressors : ndarray de forma (n_escaneos, n_neuronas)
        Un regresor por unidad, que resulta de convolucionar su curva de activación
        con la HRF glover canónica y muestrearla en frame_times.
    """
    durations = timestamps.T[1] - timestamps.T[0]
    nn_signals = []
    for amplitudes in activations.T:
        exp_condition = np.array((timestamps.T[0], durations, amplitudes))
        signal, name = compute_regressor(exp_condition, "glover", frame_times)
        nn_signals.append(signal[:,0])
    return np.array(nn_signals).T

n_runs = len(glove)
regressors = np.array([compute_regressor_from_activations(glove[run], timestamps[run], fmri_timestamps[run]) for run in range(n_runs)], dtype=object)

print("Variable: regressors")
print(f"N={n_runs}")
for run in range(n_runs):
    print(f"Tamaño de la matriz n={run} de regresores: {regressors[run].shape}")

1. ¿A que corresponde la primera dimensión?
2. ¿Y la segunda?

Veamos como luce un regresor dado:

In [ ]:
run = 0
scan = 0
regressor = regressors[run][scan]

fig, axs = plt.subplot_mosaic([['regressor', 'histogram']], layout='constrained', figsize=(9, 3))
axs["regressor"].matshow(regressor.reshape(20, 15), cmap="Spectral_r")
axs["regressor"].set_title("Regresor")
axs["regressor"].tick_params(labeltop=False, labelleft=False)
axs["histogram"].hist(regressor)
axs["histogram"].set_title("Histograma")
plt.show()

Un regresor es bastante similar al word embedding, con la diferencia que ahora es la "suma ponderada" de los embeddings de las palabras que escuchó el participante en cada escaneo de fMRI. Esta suma refleja la dinámica de una señal BOLD.

A continuación, estandarizamos los regresores y señales bold para que sus valores no indiquen valores absolutos, sino que indiquen que tanto se alejan de la media. Además, cortamos los primeros y últimos 20 segundos de cada sesión, ya que las señales de fMRI no son tan estables.

In [ ]:
def standardize(v, axis=0):
    """
    Estandariza un arreglo restando la media y dividiendo por el desvío estándar a lo largo de un eje.

    Parameters
    ----------
    v : ndarray
        Arreglo de entrada, de cualquier forma.
    axis : int, opcional
        Eje sobre el que se calculan la media y el desvío. Por defecto es 0.

    Returns
    -------
    ndarray
        Arreglo de la misma forma que v, con media 0 y desvío 1 a lo largo de axis.
    """
    return (v - np.mean(v, axis=axis, keepdims=True)) / np.std(v, axis=axis, keepdims=True)

for run in range(n_runs):
    regressors[run] = standardize(regressors[run])[10:-10]
    fmri[run] = standardize(fmri[run])[10:-10]

print("Variable: regressors")
for run in range(n_runs):
    print(f"Tamaño de la matriz n={run} de regresores: {regressors[run].shape}")

print("Variable: fmri")
for run in range(n_runs):
    print(f"Tamaño de la matriz n={run} de señales: {fmri[run].shape}")

A modo de ejemplo, prestemos atención a los valores que toman ahora los regresores y señales BOLD estandarizadas:

In [ ]:
run = 0
scan = 0
bold = fmri[run][scan]
regressor = regressors[run][scan]

fig, axs = plt.subplot_mosaic([['regressor', 'regressor-histogram'], ['bold', 'bold-histogram']], layout='constrained', figsize=(9, 6))

axs["regressor"].matshow(regressor.reshape(20, 15))
axs["regressor"].set_title("Regresor")
axs["regressor"].tick_params(labeltop=False, labelleft=False)
axs["regressor-histogram"].hist(regressor)
axs["regressor-histogram"].set_title("Histograma")

axs["bold"].matshow(bold.reshape(130, 199))
axs["bold"].set_title("Señal BOLD")
axs["bold"].tick_params(labeltop=False, labelleft=False)
axs["bold-histogram"].hist(bold)
axs["bold-histogram"].set_title("Histograma")

plt.show()

Para finalizar el paso de preparación de materiales experimentales, reunimos los conceptos que estuvimos explorando:

In [ ]:
from nilearn.plotting import plot_glass_brain

run = 1
scan = 3
bold = fmri[run][scan]
regressor = regressors[run][scan]

fig, axs = plt.subplot_mosaic([['regressor'], ['bold']], layout='constrained', figsize=(9, 6))

plot_glass_brain(masker.inverse_transform(bold), threshold=0., vmin=0., display_mode="lyrz", cmap='Spectral_r', plot_abs=True, axes=axs["bold"])
axs["bold"].set_title("Señal BOLD")

axs["regressor"].matshow(regressor.reshape(20, 15))
axs["regressor"].set_title("Regresor")
axs["regressor"].tick_params(labeltop=False, labelleft=False)
plt.show()

1. Pruebe cambiar `run` y `scan`.
2. Explique a corresponde cada variable.
4. ¿Se puede usar los regresores obtenidos a partir de *word embeddings* para predecir la señal BOLD obtenida en forma experimental en cada `run` y `scan`?

## Regresión ridge

La **regresión ridge** es una variante de la regresión lineal que agrega una penalización para evitar el sobreajuste, especialmente cuando las variables están altamente correlacionadas.

A continuación, calculamos la regresión para cada vóxel en cada sesión:

In [ ]:
import warnings
from sklearn.linear_model import Ridge

def ajustar_y_correlacionar(x_fit, y_fit, x_eval, y_eval, alpha):
    """
    Ajusta un modelo Ridge con un alpha dado y calcula la correlación de Pearson por voxel.

    Parameters
    ----------
    x_fit : ndarray de forma (n_escaneos_fit, n_regresores)
        Regresores usados para ajustar el modelo.
    y_fit : ndarray de forma (n_escaneos_fit, n_voxels)
        Señal de fMRI usada para ajustar el modelo.
    x_eval : ndarray de forma (n_escaneos_eval, n_regresores)
        Regresores de los datos sobre los que se evalúa el modelo.
    y_eval : ndarray de forma (n_escaneos_eval, n_voxels)
        Señal de fMRI real de los datos de evaluación.
    alpha : float
        Parámetro de regularización de Ridge.

    Returns
    -------
    corr : list de float, de longitud n_voxels
        Correlación de Pearson entre y_eval y la predicción, voxel por voxel.

    Notas
    -----
    Se usa fit_intercept=False porque los datos se asumen ya centrados. Si la predicción
    contiene valores no finitos, la función lanza un AssertionError.
    """
    model = Ridge(alpha=alpha, fit_intercept=False)
    model.fit(x_fit, y_fit)
    y_pred = model.predict(x_eval)
    assert np.isfinite(y_pred).all(), f"predicción no finita"
    return [np.corrcoef(y_eval[:, i], y_pred[:, i])[0, 1] for i in range(y_eval.shape[1])]


def correr_regresion_ridge(regressors, fmri, alphas=np.logspace(2, 7, 16)):
    """
    Ajusta Ridge por run con validación anidada y devuelve las correlaciones de test.

    Parámetros
    ----------
    regressors : array de objetos de longitud n_runs
        regressors[r] es de forma (n_escaneos_r, n_regresores).
    fmri : array de objetos de longitud n_runs
        fmri[r] es de forma (n_escaneos_r, n_voxels).
    alphas : ndarray
        Grilla de valores de regularización para la validación interna.

    Devuelve
    --------
    corr_runs : lista de listas
        Correlación de Pearson por run y por voxel, en el test.
    """
    warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*matmul.*")
    
    n_runs = len(fmri)
    corr_runs = []
    coef_runs = []
    for run_test in range(n_runs):
        n_voxels = fmri[run_test].shape[1]

        # Separamos el run de test: el resto son los runs de entrenamiento
        runs_train = np.setdiff1d(np.arange(n_runs), run_test)
        x_train = np.vstack([regressors[run_train] for run_train in runs_train])
        x_test = regressors[run_test]
        y_train = np.vstack([fmri[run_train] for run_train in runs_train])
        y_test = fmri[run_test]
        
        # Dentro de los runs de entrenamiento, apartamos uno para validar el alpha
        run_val = runs_train[0]
        runs_train_val = np.setdiff1d(runs_train, run_val)
        x_train_val = np.vstack([regressors[run_train_val] for run_train_val in runs_train_val])
        x_val = regressors[run_val]
        y_train_val = np.vstack([fmri[run_train] for run_train in runs_train_val])
        y_val = fmri[run_val]

        # Evaluamos cada alpha con el run de validación
        corr_val = []
        for alpha in alphas:
            corr_tmp = ajustar_y_correlacionar(x_train_val, y_train_val, x_val, y_val, alpha)
            corr_val.append(corr_tmp)

        # Nos quedamos con el alpha que da mejor correlación promedio entre voxels
        idx_best_alpha = np.argmax(np.mean(corr_val, axis=1))
        alpha = alphas[idx_best_alpha]

        # Ajustamos con todos los runs de entrenamiento y medimos en el run de test
        corr_tmp = ajustar_y_correlacionar(x_train, y_train, x_test, y_test, alpha)
        corr_runs.append(corr_tmp)

        print('run {}'.format(run_test), '\t', 'mean = {:.03f}'.format(np.mean(corr_tmp)), '\t',
              'max = {:.03f}'.format(np.max(corr_tmp)))

    print('---->', '\t' 'mean corr = {:.03f}'.format(np.mean(corr_runs)))
    return corr_runs

# Regresion ridge
corr_runs = correr_regresion_ridge(regressors, fmri)

# Medias por vóxel
means = np.mean(corr_runs, axis=0)

El código anterior imprimió la correlación promedio entre los regresores y las señales BOLD. Sin embargo, puede ser más útil ver en qué áreas se encontró mas correlación:

In [ ]:
from nilearn.plotting import plot_glass_brain

fig, axs = plt.subplot_mosaic([['nifti'], ['density']], layout='constrained', figsize=(9, 3))

plot_glass_brain(masker.inverse_transform(means), threshold=0., vmin=0., display_mode="lyrz", cmap='Spectral_r', plot_abs=True, axes=axs["nifti"])
axs["nifti"].set_title("Mapa de correlaciones")

density = sp.stats.gaussian_kde(means)
x = np.arange(0., 1, .01)
axs["density"].plot(x, density(x))
axs["density"].set_title("Gráfico de densidades")

plt.show()

La activación neural entre sujeto y sujeto es variable. Entonces, puede ser aún más útil fijarse la correlación sólo en aquellas áreas donde la activación neural entre sujetos fue más homogénea.

A continuación, esta información, que fue calculada correlacionando la actividad neural de cada sujeto a partir de los demás:

In [ ]:
from nilearn.plotting import plot_glass_brain

reliable_voxels = np.load(CARPETA / 'Practico12_reliable_voxels.npy')
print(f"Número de vóxels: {reliable_voxels.shape[0]}")

fig, axs = plt.subplot_mosaic([['nifti'], ['density']], layout='constrained', figsize=(9, 3))

plot_glass_brain(masker.inverse_transform(reliable_voxels), threshold=0., vmin=0., display_mode="lyrz", cmap='Spectral_r', plot_abs=True, axes=axs["nifti"])
axs["nifti"].set_title("Mapa de vóxels confiables")

density = sp.stats.gaussian_kde(reliable_voxels)
x = np.arange(0., 1, .01)
axs["density"].plot(x, density(x))
axs["density"].axvline(np.percentile(reliable_voxels, 75), ls='--', c='0.4')
axs["density"].set_title("Gráfico de densidades")

plt.show()

En el gráfico anterior, marcamos con una línea punteada la barrera que divide al 25% de los vóxeles más confiables. Es decir:

In [ ]:
most_reliable_voxels = reliable_voxels > np.percentile(reliable_voxels, 75)
print(f"25% de vóxels más confiables: {most_reliable_voxels.sum()}")

Finalmente, graficamos nuestro mapa de correlaciones entre embeddings y señales BOLD sólo para aquellos vóxeles que demostraron ser consistentes entre sujetos:

In [ ]:
from nilearn.plotting import plot_glass_brain

fig, axs = plt.subplot_mosaic([['nifti'], ['density']], layout='constrained', figsize=(9, 3))

plot_glass_brain(masker.inverse_transform(means * most_reliable_voxels), threshold=0., vmin=0., display_mode="lyrz", cmap='Spectral_r', plot_abs=True, axes=axs["nifti"])
axs["nifti"].set_title("Mapa de correlaciones en vóxels más confiables")

density = sp.stats.gaussian_kde(means[most_reliable_voxels])
x = np.arange(0., 1, .01)
axs["density"].plot(x, density(x))
axs["density"].set_title("Gráfico de densidades")

plt.show()

Estos valores no tienen referencia alguna, es decir, no tenemos con que compararlos.

1. ¿Contra qué podriamos compararlos para saber si esta correlación es buena o no?
2. ¿En que áreas debería haber mejor correlación?

El código a continuación calcula el promedio de activaciones en una lista de áreas de interés. Agregue cinco áreas de interés donde piense que debería haber más correlacion, y otras cinco áreas donde piense que no debería haberla, y vea como cambian las activaciones.

Para encontrar áreas puede usar un LLM como ChatGPT o Wikipedia.
Para encontrar coordenadas de algunas áreas puede usar [este mapa](https://bioimagesuiteweb.github.io/webapp/mni2tal.html) (use las coordenadas MNI).

In [ ]:
import os
import numpy as np
import nibabel as nib
from nilearn.image import iter_img
from nilearn.maskers import NiftiMasker


radius = 10

roi_dict = {'TP':(-48, 15, -27),
            'Left_PrimAud':(-48, -30, 11),
            'Broca':(-47,26,6),
            'Cerebelo':(-8,-66,-25),
           }

roi_coords = list(roi_dict.values())
roi_names = list(roi_dict.keys())


def create_roi_from_coords(coords, mask_img, radius=10):
    mask_data = mask_img.get_fdata() > 0
    ijk = np.array(np.nonzero(mask_data)).T
    xyz = nib.affines.apply_affine(mask_img.affine, ijk)

    rois = np.zeros(mask_data.shape + (len(coords),), dtype=np.float32)
    for k, c in enumerate(coords):
        d = np.linalg.norm(xyz - np.asarray(c, dtype=float), axis=1)
        sel = ijk[d <= radius]
        rois[sel[:, 0], sel[:, 1], sel[:, 2], k] = 1.

    return nib.Nifti1Image(rois, mask_img.affine)


mask_img = nib.load(CARPETA / 'Practico12_mask_lpp_en.nii')
roi_masks = create_roi_from_coords(roi_coords, mask_img, radius=radius)

nifti_masker = NiftiMasker(mask_img=mask_img, standardize=None).fit()
rois_t = nifti_masker.transform(roi_masks)
idx_rois = [np.flatnonzero(roi_t > 0.5) for roi_t in rois_t]

for idx_roi, roi_name in zip(idx_rois, roi_names):
    print(f"{roi_name}: {np.mean(means[idx_roi])}")

## Referencias

Bonnasse-Gahot, L., & Pallier, C. (2024). *Source code for "fMRI Predictors Based on Language Models of Increasing Complexity Recover Brain Left Lateralization"* (Version 1.0.0) [Computer software]. https://doi.org/10.5281/zenodo.19097232